# Day 064 — Exercise 4: Core API

Wire the components from exercises 1-3 into a FastAPI app. This is the same `build_*_api` factory pattern used throughout Section 4:

- `process_fn` injection for testability (no Ollama needed in tests)
- `initial_usage` to test rate limits without N real requests
- `ContentStore` as the in-process history backend
- `render_template` for template-based generation

In [ ]:
import re, secrets
from datetime import datetime
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

DAILY_LIMITS = {"free": 5, "pro": 500, "enterprise": float("inf")}

TEMPLATES = {
    "email":      "Write a {tone} email to {recipient} about {topic}.",
    "tweet":      "Write a {tone} tweet about {topic} in under 280 characters.",
}

def check_rate_limit(usage_count, plan):
    limit = DAILY_LIMITS.get(plan, 0)
    if usage_count >= limit:
        return False, f"Daily limit reached for {plan!r} plan"
    return True, ""

def render_template(template_str, **vars):
    required = set(re.findall(r'\{(\w+)\}', template_str))
    missing  = required - set(vars.keys())
    if missing:
        raise ValueError(f"Missing template variables: {missing}")
    result = template_str
    for k, v in vars.items():
        result = result.replace(f"{{{k}}}", str(v))
    return result

class ContentStore:
    def __init__(self):
        self._store = {}
    def add(self, user_id, prompt, content):
        cid = secrets.token_urlsafe(8)
        self._store[cid] = {"content_id": cid, "user_id": user_id,
                             "prompt": prompt, "content": content,
                             "created_at": datetime.utcnow().isoformat() + "Z"}
        return cid
    def get(self, content_id):
        return self._store.get(content_id)
    def list_user(self, user_id):
        return [v for v in self._store.values() if v["user_id"] == user_id]
    def count(self, user_id):
        return sum(1 for v in self._store.values() if v["user_id"] == user_id)


## Task

Implement `build_core_api(plan='free', process_fn=None, initial_usage=0) -> FastAPI`:

```
GET  /health              → {status, timestamp}
GET  /templates           → {templates: [...]}
POST /generate            → {content_id, content, user_id} | 429
POST /generate/template   → {content_id, content, template, user_id} | 400 | 429
GET  /history/{user_id}   → {user_id, count, items}
GET  /content/{content_id} → item | 404
```

## Your Implementation

In [ ]:
def build_core_api(plan: str = "free", process_fn=None,
                   initial_usage: int = 0) -> FastAPI:
    """FastAPI AI Writing Assistant.

    GET /health              → {status: 'ok', timestamp}
    GET /templates           → {templates: list[str]}
    POST /generate           {prompt, user_id}
                             → {content_id, content, user_id}
                             → 429 if rate limit exceeded
    POST /generate/template  {template, vars, user_id}
                             → {content_id, content, template, user_id}
                             → 400 unknown template or missing vars
                             → 429 rate limit
    GET /history/{user_id}   → {user_id, count, items}
    GET /content/{content_id} → item dict or 404

    process_fn: optional callable(prompt: str) -> str for testing.
    """
    # TODO: create app and store, add all routes
    raise NotImplementedError


In [ ]:
def build_core_api(plan: str = "free", process_fn=None,
                   initial_usage: int = 0) -> FastAPI:
    app   = FastAPI()
    store = ContentStore()
    state = {"plan": plan, "usage": initial_usage}

    class _Gen(BaseModel):
        prompt:  str = Field(min_length=1)
        user_id: str = Field(min_length=1)

    class _TGen(BaseModel):
        template: str = Field(min_length=1)
        vars:     dict = {}
        user_id:  str  = Field(min_length=1)

    @app.get("/health")
    def health():
        return {"status": "ok",
                "timestamp": datetime.utcnow().isoformat() + "Z"}

    @app.get("/templates")
    def list_templates():
        return {"templates": list(TEMPLATES.keys())}

    @app.post("/generate")
    def generate(req: _Gen):
        ok, reason = check_rate_limit(state["usage"], state["plan"])
        if not ok: raise HTTPException(429, reason)
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        state["usage"] += 1
        cid = store.add(req.user_id, req.prompt, answer)
        return {"content_id": cid, "content": answer, "user_id": req.user_id}

    @app.post("/generate/template")
    def gen_template(req: _TGen):
        ok, reason = check_rate_limit(state["usage"], state["plan"])
        if not ok: raise HTTPException(429, reason)
        tmpl = TEMPLATES.get(req.template)
        if tmpl is None: raise HTTPException(400, f"Unknown template: {req.template!r}")
        try:
            prompt = render_template(tmpl, **req.vars)
        except ValueError as e:
            raise HTTPException(400, str(e))
        answer = process_fn(prompt) if process_fn else prompt.upper()
        state["usage"] += 1
        cid = store.add(req.user_id, prompt, answer)
        return {"content_id": cid, "content": answer,
                "template": req.template, "user_id": req.user_id}

    @app.get("/history/{user_id}")
    def history(user_id: str):
        items = store.list_user(user_id)
        return {"user_id": user_id, "count": len(items), "items": items}

    @app.get("/content/{content_id}")
    def get_content(content_id: str):
        item = store.get(content_id)
        if item is None: raise HTTPException(404, f"Not found: {content_id!r}")
        return item

    return app


## Automated checks

In [ ]:
score, total = 0, 7
try:
    app = build_core_api(plan="free", process_fn=str.upper)
    c   = TestClient(app, raise_server_exceptions=False)

    # GET /health
    r = c.get("/health")
    assert r.status_code == 200 and r.json()["status"] == "ok"
    score += 1; print("\u2705 GET /health returns ok")

    # GET /templates
    rt = c.get("/templates")
    assert rt.status_code == 200
    tmpl_names = rt.json()["templates"]
    assert isinstance(tmpl_names, list) and len(tmpl_names) > 0
    score += 1; print("\u2705 GET /templates returns list of template names")

    # POST /generate
    rg = c.post("/generate", json={"prompt": "hello", "user_id": "u1"})
    assert rg.status_code == 200, f"Got {rg.status_code}: {rg.text}"
    d  = rg.json()
    assert "content_id" in d and "content" in d and d["user_id"] == "u1"
    score += 1; print("\u2705 POST /generate returns content_id + content")

    # GET /history
    rh = c.get("/history/u1")
    assert rh.status_code == 200 and rh.json()["count"] == 1
    score += 1; print("\u2705 GET /history/{user_id} returns user items")

    # GET /content/{id}
    cid = d["content_id"]
    rc  = c.get(f"/content/{cid}")
    assert rc.status_code == 200 and rc.json()["content_id"] == cid
    score += 1; print("\u2705 GET /content/{id} returns stored item")

    # 404 for unknown content_id
    r404 = c.get("/content/nonexistent_xyz")
    assert r404.status_code == 404
    score += 1; print("\u2705 GET /content/unknown \u2192 404")

    # 429 when rate limited
    at_limit = build_core_api(plan="free", process_fn=str.upper, initial_usage=5)
    cl = TestClient(at_limit, raise_server_exceptions=False)
    r429 = cl.post("/generate", json={"prompt": "x", "user_id": "u1"})
    assert r429.status_code == 429, f"Expected 429, got {r429.status_code}"
    score += 1; print("\u2705 429 when free rate limit (5) is reached")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_core_api(plan: str = "free", process_fn=None,
                   initial_usage: int = 0) -> FastAPI:
    app   = FastAPI()
    store = ContentStore()
    state = {"plan": plan, "usage": initial_usage}

    class _Gen(BaseModel):
        prompt:  str = Field(min_length=1)
        user_id: str = Field(min_length=1)

    class _TGen(BaseModel):
        template: str = Field(min_length=1)
        vars:     dict = {}
        user_id:  str  = Field(min_length=1)

    @app.get("/health")
    def health():
        return {"status": "ok",
                "timestamp": datetime.utcnow().isoformat() + "Z"}

    @app.get("/templates")
    def list_templates():
        return {"templates": list(TEMPLATES.keys())}

    @app.post("/generate")
    def generate(req: _Gen):
        ok, reason = check_rate_limit(state["usage"], state["plan"])
        if not ok: raise HTTPException(429, reason)
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        state["usage"] += 1
        cid = store.add(req.user_id, req.prompt, answer)
        return {"content_id": cid, "content": answer, "user_id": req.user_id}

    @app.post("/generate/template")
    def gen_template(req: _TGen):
        ok, reason = check_rate_limit(state["usage"], state["plan"])
        if not ok: raise HTTPException(429, reason)
        tmpl = TEMPLATES.get(req.template)
        if tmpl is None: raise HTTPException(400, f"Unknown template: {req.template!r}")
        try:
            prompt = render_template(tmpl, **req.vars)
        except ValueError as e:
            raise HTTPException(400, str(e))
        answer = process_fn(prompt) if process_fn else prompt.upper()
        state["usage"] += 1
        cid = store.add(req.user_id, prompt, answer)
        return {"content_id": cid, "content": answer,
                "template": req.template, "user_id": req.user_id}

    @app.get("/history/{user_id}")
    def history(user_id: str):
        items = store.list_user(user_id)
        return {"user_id": user_id, "count": len(items), "items": items}

    @app.get("/content/{content_id}")
    def get_content(content_id: str):
        item = store.get(content_id)
        if item is None: raise HTTPException(404, f"Not found: {content_id!r}")
        return item

    return app
```

**Architecture note**: `store` and `state` are closures inside the factory. Each `build_core_api()` call creates a fresh, independent app — perfect for tests (no shared state between test functions). In production, call `build_core_api()` once at module level and the same store persists for the lifetime of the server process.

</details>